# Original full-run artifact evidence — RUN_20260831_FULL_01

This is a read-only artifact-evidence notebook, **not a reconstructed or clean full-extraction execution transcript**. The complete extraction ran once on 2026-08-31, but the long extraction cell output was not persisted by the MCP client. This notebook preserves and verifies the original completed run through its immutable receipt, accepted page summaries, source manifests, artifact hash registers, merged CSV hash, and read-only SQLite integrity checks. It never imports or invokes the extraction runner.

In [1]:
from pathlib import Path
import sys, json, hashlib, sqlite3
import pandas as pd
from IPython.display import display, Markdown
PROJECT_ROOT=Path.cwd().resolve()
for parent in [PROJECT_ROOT,*PROJECT_ROOT.parents]:
    if (parent/"outputs"/"runs"/"RUN_20260831_FULL_01").is_dir():
        PROJECT_ROOT=parent
        break
RUN_DIR=PROJECT_ROOT/"outputs"/"runs"/"RUN_20260831_FULL_01"
receipt=json.loads((RUN_DIR/"pipeline_receipt.json").read_text(encoding="utf-8"))
assert receipt["status"]=="COMPLETE"
assert receipt["source_counts"]=={"page1":3981,"page2":3983}
print(sys.version)
print("ORIGINAL_RUN_ID =",receipt["run_id"])
print("ORIGINAL_RUN_COMPLETED_AT =",receipt["completed_at"])

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:16:45) [MSC v.1942 64 bit (AMD64)]
ORIGINAL_RUN_ID = RUN_20260831_FULL_01
ORIGINAL_RUN_COMPLETED_AT = 2026-08-31T22:32:11Z


In [2]:
page1_summary=pd.read_csv(RUN_DIR/"page1"/"run_summary.csv").iloc[0]
page2_summary=pd.read_csv(RUN_DIR/"page2"/"run_summary.csv").iloc[0]
evidence_summary=pd.DataFrame([
    {"page":"page1","engine":page1_summary.engine_version,"sources":int(page1_summary.source_count),"observations":int(page1_summary.observation_count),"failures":int(page1_summary.failure_count),"scientific_identity":page1_summary.scientific_identity_hash},
    {"page":"page2","engine":page2_summary.engine_version,"sources":int(page2_summary.source_count),"observations":int(page2_summary.observation_count),"failures":int(page2_summary.failure_count),"scientific_identity":page2_summary.scientific_identity_hash},
])
assert evidence_summary.sources.tolist()==[3981,3983]
assert evidence_summary.observations.tolist()==[129391,129472]
assert page2_summary.scientific_identity_hash=="8b48836811dcfe80fe12db79295d360f1980e54ab4e51f69f559ab87ef55d361"
display(evidence_summary)
print("ORIGINAL_FULL_RUN_PAGE_EVIDENCE = VERIFIED")

ORIGINAL_FULL_RUN_PAGE_EVIDENCE = VERIFIED


,page,engine,sources,observations,failures,scientific_identity
0,page1,page1-authoritative-0.2.3,3981,129391,0,b45bbcacadabbe53f489a6bc39323ef9f1f15181f2f900...
1,page2,page2-authoritative-1.0.0,3983,129472,0,8b48836811dcfe80fe12db79295d360f1980e54ab4e51f...


In [3]:
def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b""): digest.update(block)
    return digest.hexdigest()
verification=[]
for page in ("page1","page2"):
    register=pd.read_csv(RUN_DIR/page/"output_hashes.csv")
    for row in register.itertuples():
        artifact=RUN_DIR/page/row.artifact
        actual=sha256_file(artifact)
        verification.append({"page":page,"artifact":row.artifact,"registered_sha256":row.sha256,"actual_sha256":actual,"status":"PASS" if actual==row.sha256 else "FAIL"})
artifact_verification=pd.DataFrame(verification)
assert (artifact_verification.status=="PASS").all()
display(artifact_verification)
print("ORIGINAL_PAGE_ARTIFACT_HASHES = VERIFIED")

ORIGINAL_PAGE_ARTIFACT_HASHES = VERIFIED


,page,artifact,registered_sha256,actual_sha256,status
0,page1,checksum_source.csv.gz,49ba32ec6fdbb4cc5e5dcd7f77d344664bed060d02b683...,49ba32ec6fdbb4cc5e5dcd7f77d344664bed060d02b683...,PASS
1,page1,checksum_validation.csv.gz,313eaca331b5e1ca7c29adb5f9ae2eba98185790d2d63e...,313eaca331b5e1ca7c29adb5f9ae2eba98185790d2d63e...,PASS
2,page1,extraction_failures.csv,bc970f666620cad404b285d2c7e245206d8ea8fca95172...,bc970f666620cad404b285d2c7e245206d8ea8fca95172...,PASS
3,page1,observations.csv.gz,91e8373f36ff68b2080f54762ea9d15c041d3f9b45b217...,91e8373f36ff68b2080f54762ea9d15c041d3f9b45b217...,PASS
4,page1,page1_extraction_cleanroom.db,e212d01dd0ecf738c0840022a96765d09152f82f14a992...,e212d01dd0ecf738c0840022a96765d09152f82f14a992...,PASS
5,page1,run_summary.csv,15d8ecdcac8389014057956a0c3e6ac3ef79b209c8d850...,15d8ecdcac8389014057956a0c3e6ac3ef79b209c8d850...,PASS
6,page1,source_manifest.csv,2a0605c0074d7acbd8e8ea0352ac00d299676ae7b555cf...,2a0605c0074d7acbd8e8ea0352ac00d299676ae7b555cf...,PASS
7,page1,unresolved_values.csv,6c48841097812e21076c77ed254569fca6c469b1cfd2f1...,6c48841097812e21076c77ed254569fca6c469b1cfd2f1...,PASS
8,page1,validation_checks.csv,393e54250e6d3e37ee5e9cce610bf170a2319f4ec2b5c3...,393e54250e6d3e37ee5e9cce610bf170a2319f4ec2b5c3...,PASS
9,page1,value_lineage.csv.gz,8a423c44945f6e1113641a143370c7d6740f9197c0c944...,8a423c44945f6e1113641a143370c7d6740f9197c0c944...,PASS


In [4]:
merged_csv=RUN_DIR/"exports"/"merged_dam_daily.csv"
merged_hash=sha256_file(merged_csv)
assert merged_hash==receipt["products"]["merged_csv"]["sha256"]
database=RUN_DIR/"database"/"dam_data.db"
with sqlite3.connect(f"file:{database.as_posix()}?mode=ro",uri=True) as connection:
    integrity=connection.execute("PRAGMA integrity_check").fetchone()[0]
    foreign_keys=connection.execute("PRAGMA foreign_key_check").fetchall()
assert integrity=="ok" and not foreign_keys
final_evidence=pd.DataFrame([{
    "merged_rows":receipt["merged_row_count"],
    "merged_columns":receipt["merged_column_count"],
    "merged_csv_sha256":merged_hash,
    "database_integrity":integrity,
    "foreign_key_violations":len(foreign_keys),
    "evidence_kind":"ORIGINAL_FULL_RUN_ARTIFACT_EVIDENCE",
    "clean_extraction_transcript":False,
}])
display(final_evidence)
print("FULL_RUN_ARTIFACT_EVIDENCE_COMPLETE = TRUE")

FULL_RUN_ARTIFACT_EVIDENCE_COMPLETE = TRUE


,merged_rows,merged_columns,merged_csv_sha256,database_integrity,foreign_key_violations,evidence_kind,clean_extraction_transcript
0,129508,66,1635c09812eea94d870c87f42b1ea34cd0704a894ecff8...,ok,0,ORIGINAL_FULL_RUN_ARTIFACT_EVIDENCE,False


## Evidence interpretation

The authoritative proof of the original extraction is the completed run's mutually consistent artifacts and hashes. This notebook makes no claim that its own cells performed extraction. The refreshed executed production notebook is separately labelled presentation-only, and the GitHub release notebook has no outputs or execution counts.